# Assignment 3 — Scenario Discovery: When Do Policies Fail?

## Background

In Assignments 1 and 2 you built an exploratory ensemble and identified which uncertain
parameters most influence welfare outcomes. Now you move from *sensitivity* to
**scenario discovery**: given a policy, under what combinations of deeply uncertain
conditions does that policy fail?

In classical decision analysis, a "best" policy is chosen to maximise expected utility.
Under **deep uncertainty** — where we cannot assign reliable probabilities to future
states — the more useful question is:

> *In which corner of the uncertainty space does my policy generate unacceptable outcomes?
> Are those conditions plausible?*

This approach, called **Robust Decision Making**, seeks policies that perform adequately
across a wide range of futures rather than optimally under a single expected future.

### What you will do

You will test **three emission-control policies** (low / medium / high abatement) under
200 sampled uncertain scenarios and use **PRIM** (Patient Rule Induction Method) to
identify the parameter combinations associated with unacceptable temperature outcomes.

### Four uncertain parameters

| Parameter | Type | Range | Represents |
|-----------|------|-------|------------|
| `ssp_scenario` | integer | 0–7 | Baseline emissions pathway (SSP1-1.9 → SSP5-8.5) |
| `ecs_percentile` | continuous | 0–100 | Climate sensitivity percentile rank (cold → hot) |
| `eta` | continuous | 0.5–2.5 | Elasticity of marginal utility of consumption |
| `delta` | continuous | 0.5–2.0 | Damage function scale factor |

**Key design choice — `ecs_percentile` instead of raw `ecs_ensemble`:**  
The FaIR climate model has 1001 calibration sets, stored in a file in arbitrary order.
Using the raw index (1–1001) as a parameter is problematic for PRIM: the calibration sets
are not ordered by climate sensitivity, so index #300 is not necessarily warmer than #100.
PRIM finds compact parameter subranges; if hot calibration sets scatter randomly across
the full index, PRIM cannot find a useful subrange.

The fix: **sort** the 1001 sets by a climate sensitivity proxy (F₄ₓCO₂ / 2κ₁) and expose
the rank as `ecs_percentile` ∈ [0, 100]. Now `ecs_percentile = 0` always means the coldest
calibration set and 100 means the hottest — a physically meaningful, monotonic ordering
that PRIM can exploit.


### The JUSTICE model

JUSTICE couples three components: a **FaIR** climate module, a **Kalkuhl** damage function,
and a **Ramsey–Koopmans** welfare function. The key equations are:

**Radiative forcing** (FaIR, from atmospheric CO₂):

$$F(t) = \frac{F_{4\times CO_2}}{\ln 2}\,\ln\!\left(\frac{C_{\text{atm}}(t)}{C_{\text{pre-industrial}}}\right)$$

**ECS proxy** used to sort the 1001 FaIR calibration sets:

$$\text{ECS proxy} = \frac{F_{4\times CO_2}}{2\,\kappa_1}$$

**Damage function** (Kalkuhl, with uncertain scale $\delta$):

$$\Omega(T_t) = \delta\,a\,T_t^{\,b}$$

where $T_t$ is global mean temperature anomaly, $a$ and $b$ are empirical coefficients,
and $\delta = 1$ is the baseline estimate ($\delta > 1$ → higher damages).

**Welfare function** (Ramsey–Koopmans, with uncertain $\eta$):

$$W = \sum_{r=1}^{57}\sum_{t=2015}^{2300} L_{r,t}\;\frac{c_{r,t}^{\,1-\eta}}{1-\eta}\;e^{-\rho(t-2015)}$$

where $L_{r,t}$ is population, $\eta$ is the elasticity of marginal utility, and $\rho = 1.5\%$.

**ECR S-curve** (policy lever):

$$\tau_t = \operatorname{clip}\!\left(\frac{t-2015}{85},\,0,\,1\right), \qquad \text{ECR}(t) = \texttt{ecr\_plateau}\cdot(3\tau_t^2 - 2\tau_t^3)$$

### The SSP × ECS interaction

These two parameters create distinct structural zones in the failure space:
- **SSP0–1 at low ECS**: structural success — temperature never reaches 3°C
- **SSP6–7 at any ECS**: structural failure — policy barely reduces years above 3°C
- **SSP3–5 (policy-responsive zone)**: high abatement can eliminate failures here;
  low abatement cannot — *this is where scenario discovery finds the most interesting results*

> **Workflow note:** This assignment uses EMA Workbench `Policy` objects to run multiple
> policies in one `perform_experiments` call. The returned `experiments` DataFrame includes
> a `policy` column.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as _mpath
from IPython.display import display
from ema_workbench import (
    Model, RealParameter, IntegerParameter, ScalarOutcome, Sample,
    perform_experiments, ema_logging, MultiprocessingEvaluator,
)
from ema_workbench.em_framework.evaluators import SequentialEvaluator
from ema_workbench.analysis import prim, dimensional_stacking
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.objectives.objective_functions import years_above_temperature_threshold

os.makedirs("../plots", exist_ok=True)
ema_logging.log_to_stderr(ema_logging.INFO)

# Patch matplotlib deepcopy (required by ema_workbench)
def _path_deepcopy(self, memo=None):
    return copy.copy(self)
_mpath.Path.__deepcopy__ = _path_deepcopy

# ── ECS sorting (provided — do not modify) ──────────────────────────────────
# Sort the 1001 FaIR calibration sets by ECS proxy: F_4xCO2 / (2 * kappa1).
# This creates a physically ordered parameter: ecs_percentile=0 → coldest,
# ecs_percentile=100 → hottest. PRIM can exploit this monotonic ordering.
_calib_df = pd.read_csv(
    "../JUSTICE-0.1.4/data/input/calibrated_constrained_parameters.csv"
)
_ecs_proxy = _calib_df["clim_F_4xCO2"] / (2.0 * _calib_df["clim_kappa1"])
_sorted_orig_indices = np.argsort(_ecs_proxy.values) + 1   # 1-based

def ecs_percentile_to_ensemble(ecs_pct):
    """Convert ecs_percentile (0–100) to 1-based FaIR ensemble index, sorted by ECS."""
    rank = int(np.clip(round(float(ecs_pct) / 100.0 * 1000), 0, 1000))
    return int(_sorted_orig_indices[rank])

PARAMS          = ['ssp_scenario', 'ecs_percentile', 'eta', 'delta']
POLICY_NAMES    = ['low_abatement', 'medium_abatement', 'high_abatement']
POLICY_PLATEAUS = [0.2, 0.5, 0.8]
POLICY_COLORS   = ['#E87B4C', '#F5C842', '#4CAF7D']

## Step 1 — Define `justice_model` and run the multi-policy ensemble

**Task 1.1** — Complete `justice_model` below. The function must:
- Accept `ssp_scenario` (0–7) and `ecs_percentile` (0–100)
- Convert `ecs_percentile` to a FaIR ensemble index using `ecs_percentile_to_ensemble()`
- Set `eta` and `delta` on the model; build the S-curve ECR from `ecr_plateau`
- Return `yat_3c` (years above 3°C) and `peak_temp` (maximum temperature reached)

**Task 1.2** — Wire up the EMA model:
```python
em_model.uncertainties = [
    IntegerParameter('ssp_scenario',   0,    7   ),
    RealParameter(   'ecs_percentile', 0.0,  100.0),
    RealParameter(   'eta',            0.5,  2.5  ),
    RealParameter(   'delta',          0.5,  2.0  ),
]
em_model.levers = [RealParameter('ecr_plateau', 0.1, 0.9)]
```

**Task 1.3** — Run 200 scenarios × 3 policies using `SequentialEvaluator`.

> **Optional:** In a live Jupyter session, replace `SequentialEvaluator` with
> `MultiprocessingEvaluator(em_model, n_processes=-1)` to use all CPU cores.
> Note: this requires the model function to live in a standalone `.py` module, not a
> notebook cell (macOS spawn limitation).


In [ ]:
def justice_model(ssp_scenario=2, ecs_percentile=50.0, eta=1.45, delta=1.0,
                  ecr_plateau=0.5):
    """JUSTICE model function for EMA Workbench scenario discovery."""
    JUSTICE.hard_reset()
    scenario_idx = int(np.round(np.clip(ssp_scenario, 0, 7)))
    ensemble_idx = ecs_percentile_to_ensemble(float(ecs_percentile))

    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=scenario_idx,
        climate_ensembles=ensemble_idx,
        stochastic_run=False,
        social_welfare_function=WelfareFunction.UTILITARIAN,
    )

    ecr = np.full((model.no_of_regions, model.no_of_timesteps), ecr_plateau)
    model.run(ecr)
    data = model.evaluate()
    return {
        "yat_2c":    data["years_above_temperature_threshold"][0],
        "peak_temp": data["global_temperature"][:, 0].max(),
    }

In [ ]:
policies = [Sample(name, ecr_plateau=p)
            for name, p in zip(POLICY_NAMES, POLICY_PLATEAUS)]

with SequentialEvaluator(em_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(
        scenarios=200, policies=policies)

df_results = pd.DataFrame(outcomes)
df_results["policy"] = experiments["policy"].values

print(f"Runs: {len(df_results)}  ({len(df_results)//3} per policy)")
print(df_results.groupby("policy")[["yat_2c", "peak_temp"]].agg(["min","median","max"]).round(1).to_string())

## Step 2 — Define the failure criterion

**Failure = `yat_3c` ≥ global 75th percentile** across all 600 runs.

Why `yat_3c` (years above 3°C)?
- At 2°C, almost all scenarios exceed the threshold for the full 285-year run — too little
  variation for PRIM to discriminate. At 3°C, the SSP × ECS interaction creates rich variation.
- Unlike welfare metrics, temperature responds directly to both baseline emissions (SSP) and
  the ECR lever — the two key policy-relevant dimensions.

Why global threshold (not within-policy)?
- A within-policy 75th percentile always gives exactly 50 failures per policy (by construction),
  eliminating the policy comparison signal. A global threshold lets high abatement genuinely
  have fewer failures.

**Task 2.1** — Compute `global_thr`, the `y_by_policy` binary vectors, and print failure counts.

**Task 2.2** — Plot histograms of `yat_3c` for each policy with the threshold marked.


In [ ]:
# Choose a threshold to define "success" (low years above 2 deg C = good outcome).
# Justify your choice: too low = very few successes; too high = almost everything succeeds.
# Hint: df_results["yat_2c"].quantile(q) or a fixed physical threshold (e.g. 50 years)
global_thr = ...   # your threshold
print(f"Success threshold: {global_thr:.0f} years above 2 deg C")

y_by_policy = {}
for name in POLICY_NAMES:
    mask = df_results["policy"] == name
    y_by_policy[name] = df_results.loc[mask, "yat_2c"].values <= global_thr
    n_suc = y_by_policy[name].sum()
    print(f"  {name:20s}: {n_suc:3d}/200 successes ({100*n_suc/200:.0f}%)")

## Step 3 — PRIM for each policy

**Task 3.1** — For each policy, run PRIM:
```python
mask     = experiments['policy'] == name
x_sub    = experiments.loc[mask, PARAMS].reset_index(drop=True)
y_sub    = y_by_policy[name]
prim_alg = prim.Prim(x_sub, y_sub, threshold=0.8, peel_alpha=0.05)
box      = prim_alg.find_box()
```

**Task 3.2** — Display `box.show_tradeoff()` for each policy. You should see an arc in
coverage–density space (not a point cluster), because `ssp_scenario` creates an L-shaped
failure region. Under high abatement, the arc may flatten or disappear — what does this tell you?

**Task 3.3** — Select the knee-of-curve box:
```python
traj = box.peeling_trajectory
sel  = int((traj['coverage'] + traj['density']).idxmax())
box.select(sel)
```
Display `box.inspect(style='graph')` and record `box.coverage` and `box.density`.

> **Note on dump box:** If PRIM returns the full parameter space (no restrictions), this
> is a finding — the policy is robust enough that no compact description of failure conditions
> exists. Do not treat this as a code error.


In [ ]:
boxes = {}
for name in POLICY_NAMES:
    mask  = experiments["policy"] == name
    x_sub = experiments.loc[mask, PARAMS].reset_index(drop=True)
    y_sub = y_by_policy[name]

    # peel_alpha controls how aggressively PRIM peels; smaller = finer boxes
    prim_alg = prim.Prim(x_sub, y_sub, peel_alpha=...)   # choose a value (e.g. 0.05)
    box = prim_alg.find_box()
    boxes[name] = box

    # Select the best box by maximising coverage + density trade-off
    traj  = box.peeling_trajectory
    sel_i = max(1, min(int((traj["coverage"] + traj["density"]).idxmax()), len(traj) - 1))
    print(f"{name}: box {sel_i}  cov={traj['coverage'].iloc[sel_i]:.2f}  den={traj['density'].iloc[sel_i]:.2f}")

## Step 4 — Compare restriction fractions across policies

The **restriction fraction** of a parameter measures how tightly PRIM narrowed its range:

$$r_p = 1 - \frac{\text{box range}_p}{\text{full range}_p}$$

A value of 0 means PRIM did not restrict this parameter at all. A value of 0.9 means
PRIM reduced the allowed range to 10% of the original — this parameter is critical.

**Task 4.1** — Compute restriction fractions for all 4 parameters across all 3 policies.

**Task 4.2** — Plot as a grouped horizontal bar chart (one panel per policy).

**Task 4.3** — Which parameter is most restricted? Does `ssp_scenario` dominate, or
does `ecs_percentile` add restriction as abatement increases? Do `eta` and `delta` play
any role for the temperature-based failure criterion?


In [ ]:
param_ranges = {
    "ssp_scenario":   (0, 7),
    "ecs_percentile": (0, 100),
    "eta":            (0.5, 2.5),
    "delta":          (0.5, 2.0),
}

all_restrictions = {}
for name in POLICY_NAMES:
    box   = boxes[name]
    traj  = box.peeling_trajectory
    sel_i = max(1, min(int((traj["coverage"] + traj["density"]).idxmax()), len(traj) - 1))
    lims  = box.box_lims[sel_i]

    row = {}
    for param, (lo, hi) in param_ranges.items():
        box_lo = lims[param][0] if param in lims.columns else lo
        box_hi = lims[param][1] if param in lims.columns else hi
        row[param] = 1.0 - (box_hi - box_lo) / (hi - lo)
    all_restrictions[name] = row

df_restrictions = pd.DataFrame(all_restrictions).T
print(df_restrictions.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(param_ranges)); w = 0.25
for k, name in enumerate(POLICY_NAMES):
    ax.bar(x + k*w, df_restrictions.loc[name], width=w, label=name)
ax.set_xticks(x + w); ax.set_xticklabels(list(param_ranges.keys()))
ax.set_ylabel("Restriction fraction"); ax.legend()
ax.set_title("PRIM: fraction of parameter space restricted per policy")
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a03_restrictions.png"), dpi=120, bbox_inches="tight")
plt.show()

## Step 5 — Dimensional stacking

Dimensional stacking discretises the parameter space into bins and colours each
bin by failure rate. With `ssp_scenario` and `ecs_percentile` as the primary axes,
you can directly visualise the L-shaped failure region and how it shifts across policies.

**Task 5.1** — Apply `dimensional_stacking.create_pivot_plot` for each policy:
```python
dimensional_stacking.create_pivot_plot(x_sub, y_sub, nbins=3)
```

**Task 5.2** — Do failures cluster in the high-SSP, high-ECS corner?
Does the cluster shrink under high abatement? Are `eta` and `delta` visible
in the outer axes, or are the inner SSP/ECS axes dominant?


In [ ]:
for name in POLICY_NAMES:
    mask  = experiments["policy"] == name
    x_sub = experiments.loc[mask, PARAMS].reset_index(drop=True)
    y_sub = y_by_policy[name]

    dimensional_stacking.create_pivot_plot(x_sub, y_sub, nbins=3)
    fig = plt.gcf()
    fig.suptitle(
        f"Dimensional stacking — {name.replace('_', ' ').title()}\n"
        f"Colour = success rate (fraction below 2 deg C threshold)",
        fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", f"a03_dimstack_{name}.png"), dpi=110, bbox_inches="tight")
    plt.show()

## Step 6 — SSP × ECS failure scatter plot

To make the 2D physical failure region explicit, plot each scenario as a point in
`ssp_scenario` × `ecs_percentile` space, coloured by failure / success.

**Task 6.1** — Create a 3-panel scatter plot (one per policy):
- Green dots for success
- Coloured × markers for failure

**Task 6.2** — Describe how the failure region changes across policies:
- What is the shape of the failure region under low abatement?
- Does the "policy-responsive zone" (SSP3–5) change colour across panels?
- Can you identify the structural failure zone (SSP6–7) that persists regardless of policy?


In [ ]:
POLICY_COLORS = ["steelblue", "darkorange", "seagreen"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, name, color in zip(axes, POLICY_NAMES, POLICY_COLORS):
    mask  = experiments["policy"] == name
    x_sub = experiments.loc[mask, PARAMS].reset_index(drop=True)
    y_sub = y_by_policy[name]

    ax.scatter(x_sub.loc[~y_sub, "ssp_scenario"], x_sub.loc[~y_sub, "ecs_percentile"],
               c="#cccccc", s=30, alpha=0.7, label=f"failure  (n={(~y_sub).sum()})")
    ax.scatter(x_sub.loc[y_sub, "ssp_scenario"],  x_sub.loc[y_sub, "ecs_percentile"],
               c=color, s=55, alpha=0.9, zorder=3, label=f"success  (n={y_sub.sum()})")
    ax.set_xlabel("SSP scenario index"); ax.set_ylabel("ECS percentile")
    ax.set_title(name.replace("_", " ").title()); ax.legend(fontsize=8)
fig.suptitle("Scenario space: SSP vs. ECS — success/failure by policy", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a03_scatter_ssp_ecs.png"), dpi=120, bbox_inches="tight")
plt.show()

## Reflection Questions

**1. Box interpretation.** Describe in plain language the conditions under which each policy can keep warming below 2°C. How do the conditions change between low and high abatement?


*(Write your answer here.)*


**2. Framing the cases of interest.** Why is it more informative to search for successes rather than failures at the 2°C threshold? What would happen if you applied PRIM with failures as the cases of interest at 2°C?


*(Write your answer here.)*


**3. Coverage–density tradeoff.** Was there a visible arc in the peeling trajectory for each policy? Which policy showed the clearest tradeoff, and why? What does a short arc vs. a long arc tell you about the geometry of the success region?


*(Write your answer here.)*


**4. Feasibility of the Paris Agreement target.** Based on all the visualisations — peeling trajectories, box limits, dimensional stacking, and the SSP × ECS scatter — what can you conclude about the feasibility of keeping warming below 2°C? Under which conditions is it achievable, and what does this imply for policy design?


*(Write your answer here.)*